# 07_latent_action_world_models.ipynb – Latent Actions from Unlabeled Trajectories

This notebook demonstrates a **toy latent-action world model** in a fully reproducible Gym setting.

## Core idea

Instead of using environment actions directly, we infer a compact latent action code between consecutive states and train dynamics with that code.

## What we build

1. Collect CartPole trajectories
2. Learn encoder + latent action tokenizer + latent dynamics
3. Evaluate one-step prediction quality
4. Inspect latent code usage statistics

This is a simplified educational analogue to latent-action JEPA world model ideas.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Collect a fixed random CartPole trajectory with separate environment/action seeds.
env = gym.make("CartPole-v1")
states = []
next_states = []
n_steps = 12000

episode = 0
env.action_space.seed(42 + episode)
obs, _ = env.reset(seed=42 + episode)
for _ in range(n_steps):
    action = env.action_space.sample()
    nxt, reward, terminated, truncated, _ = env.step(action)
    states.append(obs.astype(np.float32))
    next_states.append(nxt.astype(np.float32))
    obs = nxt
    if terminated or truncated:
        episode += 1
        env.action_space.seed(42 + episode)
        obs, _ = env.reset(seed=42 + episode)

env.close()
states = torch.tensor(np.array(states), dtype=torch.float32)
next_states = torch.tensor(np.array(next_states), dtype=torch.float32)
print(states.shape, next_states.shape)
print(f"data seed starts at 42; episode {episode} uses reset/action seed {42 + episode}")

In [ ]:
# Dataset / loader
dataset = TensorDataset(states, next_states)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

state_dim = states.shape[1]
latent_dim = 32
n_codes = 8

In [ ]:
# Model components
class Encoder(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, out_dim),
        )
    def forward(self, x):
        return self.net(x)

class InferenceNet(nn.Module):
    # infer latent action code logits from (z_t, z_{t+1})
    def __init__(self, z_dim, n_codes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * z_dim, 128), nn.ReLU(),
            nn.Linear(128, n_codes),
        )
    def forward(self, z_t, z_tp1):
        return self.net(torch.cat([z_t, z_tp1], dim=-1))

class Dynamics(nn.Module):
    def __init__(self, z_dim, n_codes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim + n_codes, 128), nn.ReLU(),
            nn.Linear(128, z_dim),
        )
    def forward(self, z_t, code_onehot):
        return self.net(torch.cat([z_t, code_onehot], dim=-1))

enc = Encoder(state_dim, latent_dim).to(device)
inf_net = InferenceNet(latent_dim, n_codes).to(device)
dyn = Dynamics(latent_dim, n_codes).to(device)

params = list(enc.parameters()) + list(inf_net.parameters()) + list(dyn.parameters())
opt = torch.optim.Adam(params, lr=2e-3)

In [ ]:
# Training loop (Gumbel-Softmax for differentiable latent code sampling)
epochs = 25
temperature = 0.7
usage_counter = torch.zeros(n_codes)

for epoch in range(epochs):
    epoch_loss = 0.0
    for s_t, s_tp1 in loader:
        s_t = s_t.to(device)
        s_tp1 = s_tp1.to(device)

        z_t = enc(s_t)
        z_tp1 = enc(s_tp1).detach()

        logits = inf_net(z_t, z_tp1)
        code = F.gumbel_softmax(logits, tau=temperature, hard=True)
        z_pred = dyn(z_t, code)

        pred_loss = F.mse_loss(z_pred, z_tp1)
        code_probs = F.softmax(logits, dim=-1).mean(dim=0)
        entropy = -(code_probs * (code_probs + 1e-8).log()).sum()
        loss = pred_loss - 0.01 * entropy  # encourage balanced code usage

        opt.zero_grad()
        loss.backward()
        opt.step()

        epoch_loss += loss.item() * s_t.size(0)
        usage_counter += code.detach().cpu().sum(dim=0)

    avg_loss = epoch_loss / len(dataset)
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:02d} | loss {avg_loss:.5f}")

In [ ]:
# One-step prediction on the first 2,000 training transitions.
# The model was trained on all 12,000 transitions, so this is not held-out evaluation.
enc.eval(); inf_net.eval(); dyn.eval()
with torch.no_grad():
    s_t = states[:2000].to(device)
    s_tp1 = next_states[:2000].to(device)
    z_t = enc(s_t)
    z_tp1 = enc(s_tp1)
    logits = inf_net(z_t, z_tp1)
    code = F.one_hot(logits.argmax(dim=-1), num_classes=n_codes).float()
    z_pred = dyn(z_t, code)
    mse = F.mse_loss(z_pred, z_tp1).item()

    code_hist = code.sum(dim=0).cpu().numpy()
    code_hist = code_hist / code_hist.sum()

print(f"training-set one-step latent MSE (first 2,000 / 12,000 transitions): {mse:.6f}")
print("code usage distribution:", np.round(code_hist, 3))

## Exercises

1. Vary `n_codes` (4, 8, 16) and compare code entropy and prediction MSE.
2. Replace CartPole with another low-dimensional Gym environment.
3. Add multi-step rollout loss and compare one-step vs multi-step behavior.